In [2]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

vehicle_data = {
    "Truck": {"mileage": (3, 6), "capacity": (3000, 12000)},
    "Van": {"mileage": (9, 15), "capacity": (500, 2000)},
    "Bus": {"mileage": (3, 5), "capacity": (0, 0)},
    "Mini Truck": {"mileage": (6, 10), "capacity": (1000, 4000)}
}

rows = []

for i in range(5000):

    vehicle = random.choice(list(vehicle_data.keys()))

    fuel_type = random.choice(["Diesel","Petrol"])

    age = random.randint(1,12)

    mileage = round(
        random.uniform(
            vehicle_data[vehicle]["mileage"][0],
            vehicle_data[vehicle]["mileage"][1]
        ),
        2
    )

    max_capacity = random.randint(
        vehicle_data[vehicle]["capacity"][0],
        max(vehicle_data[vehicle]["capacity"][1],1)
    )

    if max_capacity == 0:
        cargo = 0
    else:
        cargo = random.randint(0,max_capacity)

    distance = random.randint(20,1000)

    traffic = random.choice(
        ["Low","Medium","High"]
    )

    road = random.choice(
        ["Highway","City","Mixed"]
    )

    ac = random.choice([0,1])
    
    base = distance / mileage

    load_percent = 0

    if max_capacity>0:
        load_percent = cargo/max_capacity

    multiplier = 1.0

    multiplier += load_percent*0.18

    multiplier += age*0.01

    if traffic=="Medium":
        multiplier +=0.10

    elif traffic=="High":
        multiplier +=0.22

    if road=="City":
        multiplier +=0.15

    elif road=="Mixed":
        multiplier +=0.08

    if ac:
        multiplier +=0.05

    fuel_required = base*multiplier

    fuel_required += np.random.normal(0,1.2)

    fuel_required = round(max(2,fuel_required),2)

    rows.append({

        "Vehicle_Type":vehicle,
        "Fuel_Type":fuel_type,
        "Vehicle_Age":age,
        "Average_Mileage":mileage,
        "Trip_Distance":distance,
        "Max_Capacity":max_capacity,
        "Cargo_Weight":cargo,
        "Traffic":traffic,
        "Road_Type":road,
        "AC":ac,
        "Fuel_Required":fuel_required

    })

df = pd.DataFrame(rows)

df.to_csv("fuel_dataset.csv",index=False)

print(df.head())

print(df.shape)

  Vehicle_Type Fuel_Type  Vehicle_Age  Average_Mileage  Trip_Distance  \
0        Truck    Diesel           12             3.83            774   
1   Mini Truck    Diesel            1             6.37            636   
2   Mini Truck    Diesel            8             8.36            734   
3          Van    Diesel            6             9.61            387   
4        Truck    Petrol            9             3.37            585   

   Max_Capacity  Cargo_Weight Traffic Road_Type  AC  Fuel_Required  
0          6657          1143     Low     Mixed   0         249.35  
1          1952          1034     Low     Mixed   0         118.18  
2          1026           326  Medium      City   1         126.96  
3          1278           198  Medium     Mixed   1          54.90  
4          9201          1291  Medium     Mixed   1         233.24  
(5000, 11)


In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error,r2_score


df = pd.read_csv("fuel_dataset.csv")

X = df.drop("Fuel_Required",axis=1)

y = df["Fuel_Required"]


categorical = [

"Vehicle_Type",

"Fuel_Type",

"Traffic",

"Road_Type"

]

numeric = [

"Vehicle_Age",

"Average_Mileage",

"Trip_Distance",

"Max_Capacity",

"Cargo_Weight",

"AC"

]


preprocessor = ColumnTransformer(

[

("cat",OneHotEncoder(handle_unknown="ignore"),categorical),

("num","passthrough",numeric)

]

)


model = RandomForestRegressor(

n_estimators=300,

random_state=42

)

pipeline = Pipeline(

[

("preprocessor",preprocessor),

("model",model)

]

)


X_train,X_test,y_train,y_test=train_test_split(

X,

y,

test_size=0.2,

random_state=42

)

pipeline.fit(X_train,y_train)

pred = pipeline.predict(X_test)

print("MAE :",mean_absolute_error(y_test,pred))

print("R2 :",r2_score(y_test,pred))

MAE : 6.822068533333336
R2 : 0.9867224919484023
